# MLP 공통 학습과 저장
실제 UCI 데이터의 작은 부분으로 로딩 → 학습 → 저장 → 재로딩을 연결한다. 이 노트북은 연결 검사이며 정식 성능은 reports/RESULTS_KO.md에 따로 기록한다.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open the notebook from the repository or notebooks directory")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

DATA_ROOT = Path(os.getenv("HAR_DATA_ROOT", str(ROOT / "data/raw/UCI HAR Dataset")))
if not DATA_ROOT.exists():
    raise FileNotFoundError("Run scripts/download_data.py or set HAR_DATA_ROOT")

## 사람별 분할과 데이터 계약
검증 사람 1·6·14·21·23을 고정한다. 시계열 X_seq는 (N,128,9), 가공 특징 X_features는 (N,561), y는 0~5다. 원 파일 행 번호는 1부터 세는 sample_id로 유지한다. 공식 시험 데이터는 로드하지 않는다.

In [2]:
from src.adapters.uci_reference import load_dataset
from src.contracts import prepare_data
from src.runtime import configure_runtime
from src.models import build_mlp
from src.train import train_model, predict_batches
samples, indices = load_dataset(DATA_ROOT)
print({k: len(v) for k,v in indices.items()})
config = json.loads((ROOT / "configs/E02.json").read_text())
data = prepare_data(samples, indices, config, smoke=True)
print("Smoke shapes:", data.x_fit.shape, data.x_validation.shape)

{'fit': 5577, 'validation': 1775, 'test': 0}
Smoke shapes: (60, 561) (30, 561)


## 모델과 손실
561 → 128 ReLU → 64 ReLU → 6 logits. 80,582개 파라미터. 마지막 층은 softmax를 적용하지 않으며 손실 함수에서 from_logits=True를 사용한다. 표준화는 학습 부분에서만 계산하고 검증에는 그대로 적용한다.

In [3]:
configure_runtime(config["seed"])
model = build_mlp(data.x_fit.shape[1:])
model.summary()

Model: "feature_mlp"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ hidden_1 (Dense)                │ (None, 128)            │        71,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_2 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ logits (Dense)                  │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 80,582 (314.77 KB)

 Trainable params: 80,582 (314.77 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
from datetime import datetime
config["epochs"] = 2
config["verbose"] = 0
config["run_directory"] = str(ROOT / "runs" / ("notebook_smoke_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")))
run = train_model(model, data, config)
print(json.dumps(json.loads((run / "metrics.json").read_text()), indent=2))

{
  "split": "validation",
  "task_type": "classification",
  "fit_count": 60,
  "validation_count": 30,
  "epochs_ran": 2,
  "best_epoch": 2,
  "best_val_loss": 1.7463383674621582,
  "train_seconds": 2.3466576999999234,
  "parameters": 80582,
  "test_evaluated": false,
  "smoke": true,
  "accuracy": 0.4,
  "macro_f1": 0.305668016194332,
  "recall_per_class": [
    0.4,
    0.0,
    0.8,
    1.0,
    0.2,
    0.0
  ],
  "confusion_matrix": [
    [
      2,
      0,
      3,
      0,
      0,
      0
    ],
    [
      5,
      0,
      0,
      0,
      0,
      0
    ],
    [
      1,
      0,
      4,
      0,
      0,
      0
    ],
    [
      0,
      0,
      0,
      5,
      0,
      0
    ],
    [
      0,
      0,
      0,
      4,
      1,
      0
    ],
    [
      0,
      0,
      0,
      5,
      0,
      0
    ]
  ]
}


## 원 입력에서 저장물 재사용
새 세션에서 전처리 통계와 모델을 읽고, 같은 원 입력을 변환해 동일 예측인지 확인한다. 정답은 추론 입력에 들어가지 않는다. training=False는 Dropout을 끈다.

In [5]:
import numpy as np
from tensorflow import keras
from src.adapters.preprocessing_reference import Preprocessor
pre = Preprocessor.load(run / "preprocess.npz")
restored = keras.models.load_model(run / "model.keras", compile=False)
x = pre.transform(data.raw_validation)
np.testing.assert_allclose(predict_batches(model, data.x_validation), predict_batches(restored, x), rtol=1e-5, atol=1e-6)
print("PASS: raw input -> saved preprocessing -> reloaded model")

PASS: raw input -> saved preprocessing -> reloaded model


## 공통 학습 기능
학습에서만 shuffle을 사용한다. cache는 배열로 만든 dataset의 반복 읽기를 줄이고, batch는 한 번의 갱신 단위, prefetch는 다음 배치를 준비한다. 정식 설정은 최대 40 epoch와 val_loss patience 6이며 최저 검증 손실의 가중치를 복원한다. 검증 macro F1는 여섯 행동을 같은 비중으로 평균한다.

공통 trainer는 다른 담당자의 Keras 분류 모델도 받는다. 오토인코더는 task_type='reconstruction'을 지정하고 정답 대신 입력 자체를 목표로 사용한다. 해당 모델을 구현하는 책임은 ③에게 있다.